In [1]:
import os
import joblib
import warnings
os.environ["OMP_NUM_THREADS"] = "1"
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.svm import SVR
from google.colab import files
import matplotlib.pyplot as plt
from keras.optimizers import Adam
from keras.models import Sequential
from keras.callbacks import EarlyStopping
from keras.layers import LSTM, Dense, Dropout
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from statsmodels.tsa.stattools import arma_order_select_ic
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [2]:
# Open a file upload dialog
uploaded = files.upload()

Saving Cleaned Dataset.csv to Cleaned Dataset.csv


In [3]:
# Import dataset
dataset = pd.read_csv("Cleaned Dataset.csv")
dataset

,Country,Year,BEV Percentage (Total Number Of Registrations),BEV (New Registrations),GDP,CPI,EG,Recharging Points,AC Recharging Speed (km/h),DC Recharging Speed (km/h),...,Real Range,Purchase price (EUR),Log_BEV Percentage (Total Number Of Registrations),Log_BEV (New Registrations),Log_Recharging Points,Log_GDP,Log_CPI,Log_Available,Log_DC Recharging Speed (km/h),YJ_Purchase price (EUR)
0,Austria,2011,0.18,631.0,308167.0,3.286579,1.6,963,17.500000,175.000000,...,95.000000,30469.93,0.165514,6.448889,6.871091,12.638400,1.455489,1.609438,5.170484,-1.493084
1,Austria,2012,0.13,427.0,316589.4,2.485676,1.0,1063,29.169521,217.394139,...,95.000000,30469.93,0.122218,6.059123,6.969791,12.665364,1.248662,1.609438,5.386301,-1.493084
2,Austria,2013,0.21,656.0,321191.7,2.000156,0.4,1173,48.200000,270.000000,...,142.220000,30469.93,0.190620,6.487684,7.068172,12.679797,1.098664,2.197225,5.602119,-1.493084
3,Austria,2014,0.44,1344.0,330113.5,1.605812,1.0,1393,22.600000,240.000000,...,155.770000,35555.64,0.364643,7.204149,7.239933,12.707195,0.957744,2.639057,5.484797,-0.885657
4,Austria,2015,0.55,1699.0,342083.5,0.896563,0.6,1625,47.700000,381.250000,...,215.220000,42052.00,0.438255,7.438384,7.393878,12.742813,0.640043,3.091042,5.946075,0.277809
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,Sweden,2020,9.55,27897.0,5012855.0,0.497367,-1.3,47680,39.325301,403.164557,...,286.200000,38311.38,2.356126,10.236310,10.772288,15.427516,0.403708,4.584967,6.001822,-0.451036
374,Sweden,2021,19.09,57454.0,5417760.0,2.163197,1.3,70537,40.904348,493.391304,...,308.730000,40745.01,3.000222,10.958757,11.163907,15.505193,1.151583,5.129899,6.203327,0.003737
375,Sweden,2022,32.84,94603.0,5816415.0,8.369291,3.5,83510,49.314815,594.259259,...,345.440000,43874.27,3.521644,11.457455,11.332734,15.576195,2.237437,5.537334,6.388997,0.697159
376,Sweden,2023,38.42,111338.0,6143187.0,8.548625,1.2,126758,70.307692,649.230769,...,361.794521,45999.05,3.674273,11.620335,11.750043,15.630854,2.256397,5.918894,6.477327,1.244323


In [4]:
# Define the target
target = 'Log_BEV Percentage (Total Number Of Registrations)'

In [5]:
# Load the PCA tools
tools = joblib.load("pca_processors.pkl")
scaler = tools['scaler']
pca = tools['pca_model']
features = tools['feature_names']

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator PCA from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [6]:
# Data splitting into train data and test data
pre_covid_df = dataset[dataset['Year'].between(2020, 2023)].copy()
post_covid_df = dataset[dataset['Year'] >= 2024].copy()

Please refer the jupyter notebook - **3.1.Feature Selection_BEV Percentage (Primary Target)** to understand the imputation

In [7]:
# Check which columns have NaNs and how many
print("Missing Values Per Column on Train data (2020-2023)")
print(pre_covid_df[features].isna().sum())

print("\nMissing Values Per Column Test data (2024)")
print(post_covid_df[features].isna().sum())

Missing Values Per Column on Train data (2020-2023)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           1
dtype: int64

Missing Values Per Column Test data (2024)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           0
dtype: int64


In [8]:
# Checking which rows are empty
print("Missing Log_CPI in Train data (2020-2023)")
print(pre_covid_df[pre_covid_df['Log_CPI'].isna()].reset_index()[['Country', 'Year', 'Log_CPI']])

Missing Log_CPI in Train data (2020-2023)
  Country  Year  Log_CPI
0  Greece  2020      NaN


In [9]:
# Imputation for Log_CPI NaNs by filling the spaces with the median

# Calculate the median Log_CPI for every country of train data
country_medians = pre_covid_df.groupby('Country')['Log_CPI'].median()

# Function to fill the gaps
def impute_by_country(df):
    df_clean = df.copy()

    # Loop through the data, if Log_CPI is missing, we look up that specific country's median from our 'country_medians' list.
    df_clean['Log_CPI'] = df_clean['Log_CPI'].fillna(df_clean['Country'].map(country_medians))

    return df_clean

# Apply the fix to both datasets
pre_covid_fixed = impute_by_country(pre_covid_df)
post_covid_fixed = impute_by_country(post_covid_df)

# Extract 10 features
X_train_imputed = pre_covid_fixed[features]
X_test_imputed = post_covid_fixed[features]

In [10]:
# Verfication
print("Missing Values Per Column on Train data (2020-2023)")
print(pre_covid_fixed[features].isna().sum())

print("\nMissing Values Per Column Test data (2024)")
print(post_covid_fixed[features].isna().sum())

Missing Values Per Column on Train data (2020-2023)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           0
dtype: int64

Missing Values Per Column Test data (2024)
Log_Recharging Points             0
Real Range                        0
Log_Available                     0
YJ_Purchase price (EUR)           0
AC Recharging Speed (km/h)        0
Battery Capacity                  0
Log_DC Recharging Speed (km/h)    0
EG                                0
Log_GDP                           0
Log_CPI                           0
dtype: int64


In [11]:
# Scale the features using the scaler loaded from the .pkl file
X_train_scaled = scaler.transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

In [12]:
# Transform the features into 3 Principal Components using the loaded PCA model
# and convert them back to a dataframe for easy handling
X_train_pca = pd.DataFrame(pca.transform(X_train_scaled), columns=['PC1', 'PC2'], index=pre_covid_fixed.index)
X_test_pca = pd.DataFrame(pca.transform(X_test_scaled), columns=['PC1', 'PC2'], index=post_covid_fixed.index)

In [13]:
# Define the targets
y_train = pre_covid_fixed[target]
y_test = post_covid_fixed[target]

print(f"Train Shape (PCs): {X_train_pca.shape}")
print(f"Test Shape (PCs): {X_test_pca.shape}")

Train Shape (PCs): (108, 2)
Test Shape (PCs): (27, 2)


In [14]:
# Define the clusters based on the K-Means clusters + PCA
leaders_list = [
    'Austria', 'Belgium', 'Czech Republic', 'Denmark', 'Finland',
    'France', 'Germany', 'Hungary', 'Italy', 'Netherlands',
    'Poland', 'Portugal', 'Romania', 'Spain', 'Sweden'
]

followers_list = [
    'Bulgaria', 'Croatia', 'Cyprus', 'Estonia', 'Greece',
    'Ireland', 'Latvia', 'Lithuania', 'Luxembourg', 'Malta',
    'Slovakia', 'Slovenia'
]

In [15]:
# Filter Training Data
X_train_leaders = X_train_pca[pre_covid_fixed['Country'].isin(leaders_list)]
y_train_leaders = y_train[pre_covid_fixed['Country'].isin(leaders_list)]

X_train_followers = X_train_pca[pre_covid_fixed['Country'].isin(followers_list)]
y_train_followers = y_train[pre_covid_fixed['Country'].isin(followers_list)]

# Filter Testing Data
X_test_leaders = X_test_pca[post_covid_fixed['Country'].isin(leaders_list)]
y_test_leaders = y_test[post_covid_fixed['Country'].isin(leaders_list)]

X_test_followers = X_test_pca[post_covid_fixed['Country'].isin(followers_list)]
y_test_followers = y_test[post_covid_fixed['Country'].isin(followers_list)]

## Proposed Model

**1. LSTM Model**

In [16]:
def run_lstm_model(X_train, y_train, X_test, y_test, country_name):

    # Scale features and target that shrinks the data between 0 and 1.
    scaler_X = MinMaxScaler()
    scaler_y = MinMaxScaler()

    # Applies the same transformation for train and test data for features and target
    # For features
    X_train_scaled = scaler_X.fit_transform(X_train)
    X_test_scaled = scaler_X.transform(X_test)

    # For target
    # The scaler expects a 2D array, so we turn a flat list of targets into a column.
    y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
    y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

    # Reshape for LSTM (samples, timestamp, features) for training
    # By timestamps = 1, to tell the model that each prediction is based on the current year's datapoint
    X_train_3D = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
    X_test_3D = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

    # LSTM Architecture
    model = Sequential([
        LSTM(16, activation='tanh', input_shape=(1, X_train.shape[1])), # 16 neurons, standard tanh activation
        Dropout(0.05), # Dropout of 5% neurons during training to prevent overfitting and prevent model from learning too much from training dataset
        Dense(8, activation='relu'), # Hidden layer with 8 neurons and relu activation function
        Dense(1) # Output layer with 1 neuron
    ])

    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse') # model tries to minimise the mse, i.e, square of the difference b/w actual and predicted values

    # Early stopping to prevent overfittting with patience level of 20, i.e. if the loss doesnt improve in 20 epochs the training stops to save time.
    early_stop = EarlyStopping(
        monitor='loss', patience=20, restore_best_weights=True
    )

    # Fit the model training data
    # No validation set because the dataset is less.
    model.fit(X_train_3D, y_train_scaled,
              epochs=200, # 200 epochs
              batch_size=2, # The model updates its weights after looking at every 2 rows of data.
              verbose=0,
              callbacks=[early_stop])

    # Predict on train data
    train_pred_scaled = model.predict(X_train_3D)
    train_pred_real = np.exp(scaler_y.inverse_transform(train_pred_scaled).flatten())  # Transform prediction results back to the original scale
    y_train_real = np.exp(y_train.values) # Transform actual results back to the original scale
    train_years = pre_covid_fixed[pre_covid_fixed['Country'] == country_name]['Year'].values # Extract years for training set

    # Predict on test data
    test_pred_scaled = model.predict(X_test_3D)
    test_pred_real = np.exp(scaler_y.inverse_transform(test_pred_scaled).flatten()) # Transform prediction results back to the original scale
    y_test_real = np.exp(y_test.values) # Transform actual results back to the original scale
    test_years = post_covid_fixed[post_covid_fixed['Country'] == country_name]['Year'].values # Extract years for test set

    # We pull the years directly from the dataframes used to split your data
    train_years = pre_covid_fixed[pre_covid_fixed['Country'] == country_name]['Year'].values
    test_years = post_covid_fixed[post_covid_fixed['Country'] == country_name]['Year'].values

    # Return Train DataFrame
    df_train_lstm = pd.DataFrame({
        'Country': country_name,
        'Year': train_years,
        'Actual_BEV_%': y_train_real,
        'LSTM_Pred': train_pred_real,
        'Type': 'Train'
    })

    # Return Test DataFrame
    df_test_lstm =  pd.DataFrame({
        'Country': country_name,
        'Year': post_covid_fixed[post_covid_fixed['Country'] == country_name]['Year'].values,
        'Actual_BEV_%': y_test_real,
        'LSTM_Pred': test_pred_real
    })

    return pd.concat([df_train_lstm, df_test_lstm], ignore_index=True)

**2. ARIMAX Model**

In [17]:
def run_arimax_model(X_train, y_train, X_test, y_test, country_name):

    # Auto-select AR and MA order
    # where P "Autoregressive": "How many past years should we look at to predict the next year"
    # where q "Moving Average": "How many past errors/trends should the model consider"
    try:
        order_res = arma_order_select_ic(y_train, ic='aic', max_ar=2, max_ma=2)

        # aic: Akaike Information Criterion: A statistical estimator of prediction error between p and q.
        # It tries different combinations of p and q, and picks the combination with the lowest aic.
        # Lower the aic better the prediction. It a point where the model is neither overfitting noe underfitting
        p, q = order_res.aic_min_order
    except Exception:
        p, q = 1, 0  # fallback meaning that if the "try" block fails it will initialise p=1 and q=0

    # Train the model on the X-ogenous pca features
    # 1=d(Integrated) component, which subtracts the previous year from the current year to make the data stable before analyzing it.
    model = ARIMA(y_train, exog=X_train, order=(p, 1, q)).fit()

    # Predict on test data
    train_pred_log = model.predict(start=0, end=len(y_train)-1, exog=X_train)
    train_p_real = np.exp(train_pred_log) # Transform prediction results back to the original scale
    y_train_real = np.exp(y_train) # Transform prediction results back to the original scale
    train_years = pre_covid_fixed[pre_covid_fixed['Country'] == country_name]['Year'].values # Extract training years

    # Predict on test data
    test_pred_log = model.forecast(steps=len(y_test), exog=X_test)
    test_p_real = np.exp(test_pred_log) # Transform prediction results back to the original scale
    y_test_real = np.exp(y_test) # Transform prediction results back to the original scale
    test_years = post_covid_fixed[post_covid_fixed['Country'] == country_name]['Year'].values # Extract testing years

    # Return Train DataFrame
    df_train_arimax = pd.DataFrame({
        'Country': country_name,
        'Year': train_years,
        'Actual_BEV_%': y_train_real.values,
        'ARIMAX_Pred': train_p_real.values,
        'Type': 'Train'
    })

    # Return Test DataFrame
    df_test_arimax = pd.DataFrame({
        'Country': country_name,
        'Year': post_covid_fixed[post_covid_fixed['Country'] == country_name]['Year'].values,
        'Actual_BEV_%': y_test_real.values,
        'ARIMAX_Pred': test_p_real.values
    })

    return pd.concat([df_train_arimax, df_test_arimax], ignore_index=True)

In [18]:
# Combining leaders and followers datasets
all_meta_input = []

for country in (leaders_list + followers_list):

    # Data for the specific country to train and test datframe
    # X_tr = train using features from leaders + followers, y_tr = target of leaders + followers
    # X_te = test on using features from leaders + followers, y_te = target of leaders + followers
    X_tr = X_train_pca[pre_covid_fixed['Country'] == country].reset_index(drop=True)
    y_tr = y_train[pre_covid_fixed['Country'] == country].reset_index(drop=True)
    X_te = X_test_pca[post_covid_fixed['Country'] == country].reset_index(drop=True)
    y_te = y_test[post_covid_fixed['Country'] == country].reset_index(drop=True)

    # Run both models
    df_arimax = run_arimax_model(X_tr, y_tr, X_te, y_te, country)
    df_lstm = run_lstm_model(X_tr, y_tr, X_te, y_te, country)

    # Merge and store the prediction results for as per "Country" and "Year"
    merged = pd.merge(df_arimax, df_lstm[['Country', 'Year', 'LSTM_Pred']], on=['Country', 'Year'])
    all_meta_input.append(merged)

# Prepare the input data for Meta Model
meta_dataset = pd.concat(all_meta_input, ignore_index=True)

# Label all the rows with "Year" = 2024 as 'Test', else 'Train'
meta_dataset['Type'] = np.where(meta_dataset['Year'] == 2024, 'Test', 'Train')

# Add labels to the countries if it belongs to the Leaders or Followers group
meta_dataset['Cluster'] = np.where(meta_dataset['Country'].isin(leaders_list), 'Leaders', 'Followers')

# Sort in alphabetical order and year-wise
cols = ['Country', 'Year', 'Actual_BEV_%', 'Type', 'ARIMAX_Pred', 'LSTM_Pred', 'Cluster']
meta_dataset = meta_dataset[cols].sort_values(['Country', 'Year']).reset_index(drop=True)


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/lo

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/lo

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 139ms/step


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/lo

...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 134ms/step


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/lo

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/lo

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 129ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step


In [19]:
meta_dataset.head()

,Country,Year,Actual_BEV_%,Type,ARIMAX_Pred,LSTM_Pred,Cluster
0,Austria,2020,7.41,Train,0.393717,7.595047,Leaders
1,Austria,2021,14.92,Train,12.735234,14.778637,Leaders
2,Austria,2022,16.70,Train,21.479141,19.307398,Leaders
3,Austria,2023,20.91,Train,17.219704,17.083080,Leaders
4,Austria,2024,18.58,Test,21.754511,16.152918,Leaders


**3. META-MODEL (ElasticNet)**

In [20]:
from sklearn.linear_model import ElasticNetCV

final_results_list = []

# Create a dictionary to hold the meta-models for each country
saved_meta_models = {}

for cluster in meta_dataset['Cluster'].unique():
    cluster_df = meta_dataset[meta_dataset['Cluster'] == cluster].copy()
    train = cluster_df[cluster_df['Type'] == 'Train']
    if train.empty: continue

    # Initialise ElasticNet
    model_enet = ElasticNetCV(
        l1_ratio=[.1, .5, .7, .9, .95, .99, 1],
        alphas=np.logspace(-3, 3, 20),
        cv=3,
        positive=True,
        max_iter=50000
    )

    # Fit on ARIMAX and LSTM predictions
    model_enet.fit(train[['ARIMAX_Pred', 'LSTM_Pred']], train['Actual_BEV_%'])

    # Predict for each country in the cluster
    for country in cluster_df['Country'].unique():
        country_df = cluster_df[cluster_df['Country'] == country].copy()
        test = country_df[country_df['Type'] == 'Test']
        if test.empty: continue

        pred = model_enet.predict(test[['ARIMAX_Pred', 'LSTM_Pred']])[0]

        res = test.copy()
        res['Stacked_Pred'] = np.clip(pred, 0, 100)
        res['Error_Abs'] = abs(res['Actual_BEV_%'] - res['Stacked_Pred'])
        final_results_list.append(res)

# Combine results into one DataFrame
meta_results = pd.concat(final_results_list, ignore_index=True)


In [21]:
all_leaders = []
all_followers = []

for year in sorted(meta_results['Year'].unique()):
    year_data = meta_results[meta_results['Year'] == year]

    # Leaders
    leaders_data = year_data[year_data['Cluster'] == 'Leaders'] \
        .sort_values(['Country', 'Year'], ascending=[True, True])

    # Followers
    followers_data = year_data[year_data['Cluster'] == 'Followers'] \
        .sort_values(['Country', 'Year'], ascending=[True, True])

    # Store
    all_leaders.append(leaders_data)
    all_followers.append(followers_data)

# Combine all years and sort by Country then Year
leaders_final = pd.concat(all_leaders).sort_values(['Country', 'Year'], ascending=[True, True])
followers_final = pd.concat(all_followers).sort_values(['Country', 'Year'], ascending=[True, True])

In [22]:
# Save Excel files
leaders_final.to_excel("Trial3_Proposed (ElasticNet)_Leaders_Results.xlsx", index=False)
followers_final.to_excel("Trial3_Proposed (ElasticNet)_Followers_Results.xlsx", index=False)